In [ ]:



from google.colab import drive
import os
import logging

drive.mount('/content/drive')

# Configure Logging ---dfdfdfdf
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
logging.info("Google Drive mounted.")

#update to your actual folders/files ----
BASE_TXT_DIR = "/content/drive/MyDrive/Projects/Upwork/data"
ARXIV_JSON_PATH = "/content/drive/MyDrive/Projects/Upwork/arxiv-metadata-oai-snapshot-2-001.json"

# Define Output Directories ---
OUTPUT_BASE_DIR = "/content/drive/MyDrive/Projects/Upwork/outputs" # Base output directory

# Create base output directory if it doesn't exist
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

# Define specific output directories
DRIFT_JSON_DIR = os.path.join(OUTPUT_BASE_DIR, "drift_json")
TOPIC_MODEL_DIR = os.path.join(OUTPUT_BASE_DIR, "topic_models")
VALIDATION_REPORT_DIR = os.path.join(OUTPUT_BASE_DIR, "validation_reports")
PLOT_DIR = os.path.join(OUTPUT_BASE_DIR, "plots") # <--- Added PLOT_DIR definition

# Create specific directories if they don't exist
os.makedirs(DRIFT_JSON_DIR, exist_ok=True)
os.makedirs(TOPIC_MODEL_DIR, exist_ok=True)
os.makedirs(VALIDATION_REPORT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True) # <--- Added directory creation

logging.info(f"Base text directory: {BASE_TXT_DIR}")
logging.info(f"ArXiv JSON path: {ARXIV_JSON_PATH}")
logging.info(f"Output directories setup under: {OUTPUT_BASE_DIR}")


#Analysis parameters ----
SEED_WORD = "cloud"
ANCHOR_DEFINITION = (
    "networked computing facilities providing remote data storage and "
    "processing services (typically via the internet), considered collectively. "
    "Also as count noun: a particular facility of this type."
)
REFERENCE_YEAR = 2007             # Year for W2V drift comparison baseline
GENRES_TO_USE = ["academicwriting", "fiction", "newspaper", "magazine"]  # or None for all
TOPK_NEIGHBORS = 15                 # For W2V neighborhood overlap & fuzziness
MIN_YEAR_TXT = 1800                 # Lower bound for .txt files
MIN_YEAR_ARXIV = 1990               # Lower bound for arXiv files
ARXIV_MAX_ENTRIES = None            # Set to a number (e.g., 40000) for faster testing, None for all
RANDOM_SEED = 42                    # For reproducibility in W2V and sampling

# Word2Vec Parameters
W2V_VECTOR_SIZE = 200
W2V_MIN_COUNT = 8
W2V_WINDOW = 5
W2V_EPOCHS = 10
W2V_WORKERS = 4 # Use multiple cores if available

# BERT Parameters
BERT_MODEL_NAME = "all-MiniLM-L6-v2"
BERT_BATCH_SIZE = 64

# Validation Pipeline Parameters (Ensure these are set logically based on analysis)
SHIFT_YEAR_1 = 2008 # Example first shift year
SHIFT_YEAR_2 = 2011 # Example second shift year
VAL_WINDOW_HALF_WIDTH = 2
BERTOPIC_MIN_SIZE = 5
MLM_MODEL_NAME = "bert-base-uncased"
MLM_SENTENCES_PER_YEAR = 10
MLM_RANK_OVERLAP_K = 10
MLM_BOOTSTRAP_SAMPLES = 50 # Reduced for memory/speed

logging.info(f"Analysis Parameters Set: Seed='{SEED_WORD}', RefYear={REFERENCE_YEAR}, Validation Span=({SHIFT_YEAR_1}, {SHIFT_YEAR_2})")

Mounted at /content/drive


In [ ]:
#cell2
import os, re, json, math, numpy as np, matplotlib.pyplot as plt
from collections import defaultdict, Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("stopwords")
STOP = set(stopwords.words("english"))

def simple_clean(text: str):
    """Lowercase, tokenize, keep alphabetic tokens >2 chars and not in stopwords."""
    toks = word_tokenize(text.lower())
    toks = [t for t in toks if t.isalpha() and len(t) > 2 and t not in STOP]
    return toks

def find_year_in_string(s: str):
    m = re.search(r"\b(18|19|20)\d{2}\b", s)
    return int(m.group()) if m else None

def year_from_arxiv_record(obj: dict):
    # Prefer update_date, else first version.created, else fallback from id prefix
    if obj.get("update_date"):
        s = obj["update_date"]
        if re.match(r"\d{4}-\d{2}-\d{2}", s):
            return int(s[:4])
    if obj.get("versions"):
        created = obj["versions"][0].get("created", "")
        m = re.search(r"\b(19|20)\d{2}\b", created)
        if m: return int(m.group())
    # id like "0704.0001" -> 2007
    if "id" in obj:
        m = re.match(r"^(\d{2})\d{2}\.", obj["id"])
        if m:
            yy = int(m.group(1))
            return 1900+yy if yy >= 90 else 2000+yy
    return None


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
#cell3
def load_txt_by_genre_year(base_dir, genres=None, min_year=1800):
    """
    Returns dict: genre -> {year(str): [doc, ...]}
    Assumes filenames contain a 4-digit year: e.g., "w_acad_1997.txt".
    """
    if genres is None:
        genres = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    out = {}
    for g in genres:
        gpath = os.path.join(base_dir, g)
        if not os.path.isdir(gpath):
            continue
        yearwise = defaultdict(list)
        for fname in os.listdir(gpath):
            if not fname.endswith(".txt"):
                continue
            y = find_year_in_string(fname)
            if y is None or y < min_year:
                continue
            fpath = os.path.join(gpath, fname)
            try:
                txt = open(fpath, "r", encoding="utf-8", errors="ignore").read()
            except Exception:
                continue
            toks = simple_clean(txt)
            if toks:
                yearwise[str(y)].append(" ".join(toks))
        out[g] = dict(yearwise)
    return out

txt_data = load_txt_by_genre_year(BASE_TXT_DIR, GENRES_TO_USE, MIN_YEAR_TXT)
print("Loaded TXT genres:", list(txt_data.keys()))


Loaded TXT genres: ['academicwriting', 'fiction', 'newspaper', 'magazine']


In [ ]:
#cell4
import nltk
nltk.download('punkt_tab')
def load_arxiv_by_year(json_path, max_entries=None, min_year=1990):
    """
    Streams a JSONL file: each line is a JSON object.
    Returns dict: {year(str): [doc_str, ...]}
    """
    yearwise = defaultdict(list)
    with open(json_path, "r") as f:
        for i, line in enumerate(f):
            try:
                obj = json.loads(line)
            except Exception:
                continue
            y = year_from_arxiv_record(obj)
            if y is None or y < min_year:
                continue
            text = (obj.get("title","") + " " + obj.get("abstract","")).strip()
            toks = simple_clean(text)
            if toks:
                yearwise[str(y)].append(" ".join(toks))
            if max_entries and (i+1) >= max_entries:
                break
    return dict(yearwise)

# For first run, you can limit max_entries (e.g., 10000) to sanity-check quickly....change max entries
arxiv_years = load_arxiv_by_year(ARXIV_JSON_PATH, max_entries=40000, min_year=MIN_YEAR_ARXIV)
print("arXiv years (sample):", list(arxiv_years.keys())[:50])


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


arXiv years (sample): ['2008', '2007', '2013', '2015', '2009', '2010', '2014', '2022', '2011', '2012', '2016', '2021', '2019', '2023', '2017', '2024', '2020', '2018', '2025']


In [ ]:
#cell5
def merge_yearwise(txt_by_genre, arxiv_by_year):
    merged = defaultdict(list)
    # TXT
    for genre, ydict in txt_by_genre.items():
        for y, docs in ydict.items():
            merged[y].extend(docs)
    # arXiv
    for y, docs in arxiv_by_year.items():
        merged[y].extend(docs)
    # sort years as strings
    return {y: merged[y] for y in sorted(merged.keys(), key=lambda s: int(s))}

year_docs = merge_yearwise(txt_data, arxiv_years)
print("Years available:", list(year_docs.keys())[:15])
print("Docs in ref year {}:".format(REFERENCE_YEAR), len(year_docs.get(str(REFERENCE_YEAR), [])))


Years available: ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021']
Docs in ref year 2007: 11803


In [ ]:
#cell6
!pip install -q gensim
from gensim.models import Word2Vec
import logging # Use logging instead of print for better control

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def train_w2v_per_year(year_docs, vector_size=200, window=5, min_count=10,
                       epochs=10, min_unique=200, workers=4, seed=42): # Added seed, increased default workers
    """
    Builds vocab first, prints diagnostics, skips years with too-small vocab.
    - min_unique: minimum unique tokens required to train for a year
    - seed:       Fixed seed for reproducibility.
    """
    models = {}
    logging.info("Starting Word2Vec model training for each year...")
    for y, docs in sorted(year_docs.items(), key=lambda kv: int(kv[0])):
        # No 'only_from_year' filter here ..............., train for all available years
        if len(docs) < 3:
            logging.warning(f"[{y}] Skipped: too few docs ({len(docs)})")
            continue

        sentences = [d.split() for d in docs]
        total_tokens = sum(len(s) for s in sentences)
        if total_tokens < 1000:
            logging.warning(f"[{y}] Skipped: too few tokens ({total_tokens})")
            continue

        # --- MODIFICATION: Added seed=seed, workers=workers ---
        model = Word2Vec(vector_size=vector_size, window=window,
                         min_count=min_count, workers=workers, sg=1, seed=seed)

        model.build_vocab(sentences)
        kept = len(model.wv.key_to_index)
        logging.info(f"[{y}] Docs={len(docs)}, Tokens≈{total_tokens} | Kept Vocab={kept} (min_count={min_count})")

        if kept < min_unique:
            logging.warning(f"[{y}] Skipped: Kept vocab < {min_unique}. Try lowering min_count or merging slices.")
            continue

        model.train(sentences, total_examples=len(sentences), epochs=epochs)
        models[y] = model
        logging.info(f"[{y}] Trained Word2Vec model ({epochs} epochs).")

    logging.info("Finished Word2Vec model training.")
    return models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 33.5 MB/s eta 0:00:00


In [ ]:
##cell 8
W2V_EPOCHS = 10

# --- MODIFICATION: Removed only_from_year from this call ---
# Train models for ALL years available in year_docs.
# Filtering relative to REFERENCE_YEAR happens later during drift calculation.
logging.info(f"Training Word2Vec models starting from the earliest available year up to the latest...")
w2v_models = train_w2v_per_year(
    year_docs,
    vector_size=W2V_VECTOR_SIZE,
    window=W2V_WINDOW,
    min_count=W2V_MIN_COUNT,
    epochs=W2V_EPOCHS,
    min_unique=200,
    workers=4, # Use multiple workers for speed
    seed=42    # Ensure reproducibility
)

print("\nTrained W2V models for years:", list(w2v_models.keys())) # Use print for final summary


Trained W2V models for years: ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2019']


In [ ]:
# Cell 9: W2V Anchor Vector and Neighbors Function

import numpy as np
import logging

def w2v_anchor_and_neighbors(models, ref_year, seed, definition, topk=15):
    """
    Anchor vector from definition tokens present in ref year's vocab; fallback to seed.
    Get top-k neighbors based on the anchor vector derived from the definition.
    """
    ref_year_str = str(ref_year)
    if ref_year_str not in models:
        logging.error(f"Reference year {ref_year_str} not found in trained W2V models.")
        raise ValueError(f"Reference year {ref_year_str} not found in trained W2V models.")
    ref_wv = models[ref_year_str].wv
    logging.info(f"Generating W2V anchor vector and neighbors for ref year {ref_year_str} using definition.")

    # Calculate anchor vector from definition
    def_tokens = simple_clean(definition) # Assumes simple_clean is defined in an earlier cell
    valid_def_tokens = [t for t in def_tokens if t in ref_wv.key_to_index]

    anchor_vec = None
    if valid_def_tokens:
        anchor_vec = np.mean([ref_wv[t] for t in valid_def_tokens], axis=0)
        logging.info(f"W2V anchor vector derived from {len(valid_def_tokens)} definition tokens.")
    elif seed in ref_wv.key_to_index:
        anchor_vec = ref_wv[seed]
        logging.warning(f"W2V: No definition tokens found in vocab for year {ref_year_str}. Falling back to seed word '{seed}' for anchor vector.")
    else:
        logging.error(f"W2V: Neither definition tokens nor seed word '{seed}' exist in ref year {ref_year_str} vocab.")
        raise ValueError(f"Neither definition tokens nor seed word '{seed}' exist in reference year {ref_year_str} vocab.")

    # --- MAJOR FIX START: Calculate neighbors based on the anchor_vec ---
    neigh = []
    if anchor_vec is not None:
        try:
            # Find words most similar to the calculated anchor_vec
            # Get a few extra to allow filtering the seed/definition words
            similar_words = ref_wv.most_similar(positive=[anchor_vec], topn=topk + 5)
            # Filter out the original seed word and definition tokens
            neigh = [w for w, _ in similar_words if w != seed and w not in valid_def_tokens][:topk]
            logging.info(f"Found {len(neigh)} W2V neighbors based on the definition anchor vector: {neigh}")
        except Exception as e:
            logging.error(f"W2V: Could not find neighbors for the anchor vector in year {ref_year_str}: {e}")
    else:
        logging.error(f"W2V: Anchor vector is None, cannot find neighbors.")
    # --- MAJOR FIX END ---

    return anchor_vec, neigh

In [ ]:
# Cell 10: Alignment Helper Functions & Membership Helpers

import numpy as np
from numpy.linalg import svd
import logging

# --- Alignment Helpers ---
def intersect_vocab(ref_wv, tgt_wv, topn=None):
    """Returns shared vocab between two KeyedVectors, ranked by reference frequency."""
    try:
        shared = set(ref_wv.key_to_index).intersection(tgt_wv.key_to_index)
        if not shared:
            logging.warning("Intersection vocab: No shared words found between models.")
            return []
        ranked = [w for w in ref_wv.index_to_key if w in shared]
        if topn is not None:
            ranked = ranked[:topn]
        return ranked
    except Exception as e:
        logging.error(f"Error intersecting vocab: {e}")
        return []

def procrustes_R(ref_wv, tgt_wv, anchor_words, dim=None):
    """Orthogonal Procrustes rotation aligning target (Y) to reference (X) using anchor_words."""
    if not anchor_words:
        raise ValueError("Anchor words list is empty for Procrustes alignment.")
    logging.debug(f"Performing Procrustes alignment using {len(anchor_words)} anchor words.")
    try:
        valid_anchors = [w for w in anchor_words if w in ref_wv.key_to_index and w in tgt_wv.key_to_index]
        if len(valid_anchors) < len(anchor_words):
             logging.warning(f"Procrustes: Using {len(valid_anchors)}/{len(anchor_words)} valid anchor words.")
        if len(valid_anchors) < 10:
             raise ValueError(f"Too few valid anchor words ({len(valid_anchors)}) for stable Procrustes alignment.")

        X = np.stack([ref_wv[w] for w in valid_anchors])
        Y = np.stack([tgt_wv[w] for w in valid_anchors])
        X = X - X.mean(axis=0, keepdims=True)
        Y = Y - Y.mean(axis=0, keepdims=True)
        U, _, Vt = svd(Y.T @ X, full_matrices=False)
        R = U @ Vt
        want_dim = dim if dim is not None else ref_wv.vector_size
        if R.shape != (want_dim, want_dim):
            logging.warning(f"Procrustes R shape mismatch ({R.shape} vs {(want_dim, want_dim)}). Returning identity.")
            R = np.eye(want_dim)
        return R
    except KeyError as e:
        raise ValueError(f"KeyError during Procrustes: Word '{e}' not found.")
    except Exception as e:
        logging.error(f"Error during SVD/Procrustes calculation: {e}", exc_info=True)
        raise RuntimeError(f"SVD/Procrustes failed: {e}")

# --- MODIFICATION START: Added Membership Helper Functions ---
def categorize_membership(fuzz):
    """Categorizes fuzziness score into stable, transitional, unstable."""
    if fuzz is None: return "n/a"
    try:
        f = float(fuzz)
        if f >= 0.70: return "stable"
        if f >= 0.40: return "transitional"
        return "unstable"
    except (ValueError, TypeError):
        logging.warning(f"Could not convert fuzziness '{fuzz}' to float for categorization.")
        return "error"

def uncertainty_from_cat(cat):
    """Determines uncertainty level based on membership category."""
    return {"stable":"low", "transitional":"medium", "n/a":"n/a", "error":"error"}.get(cat, "provisional")
# --- MODIFICATION END ---

print("Alignment and Membership helper functions defined.") # Add print confirmation

Alignment and Membership helper functions defined.


In [ ]:
# Cell 11: W2V Drift Calculation, Plotting, and JSON Output

from scipy.spatial.distance import cosine
import numpy as np
import logging
import matplotlib.pyplot as plt # Import plotting library
import json # Import json library
from datetime import datetime # Import datetime
import os # Import os for path joining

# (Ensure PLOT_DIR and DRIFT_JSON_DIR are defined from Cell 1)
# (Ensure membership helper functions are defined in Cell 10)

# --- W2V Drift Calculation Function ---
def compute_w2v_drift(models, seed, anchor_vec, anchor_neighbors, ref_year, topn_vocab=20000):
    """
    Aligns each year's space to ref via Procrustes (using shared vocab),
    then computes drift metrics starting FROM the reference year.
    Uses definition-based anchor_vec and anchor_neighbors.
    """
    ref_year_str = str(ref_year)
    if ref_year_str not in models:
        logging.error(f"Reference year {ref_year_str} not found in trained W2V models.")
        raise ValueError(f"Reference year {ref_year_str} not found.")

    ref_wv = models[ref_year_str].wv
    dim = ref_wv.vector_size
    results = []
    logging.info(f"Computing Word2Vec drift relative to reference year {ref_year_str}...")
    stable_anchor_words = [w for w in ref_wv.index_to_key[:topn_vocab] if w in ref_wv.key_to_index]
    min_align_words = 50 # Minimum shared words needed for stable alignment
    if len(stable_anchor_words) < min_align_words:
        logging.warning(f"Found only {len(stable_anchor_words)} potential stable anchor words in reference year. Alignment might be unstable.")


    for y_str in sorted(models.keys(), key=int):
        # --- Filter: Calculate drift only from reference year onwards ---
        if int(y_str) < int(ref_year_str):
            continue
        # --- End Filter ---

        tgt_wv = models[y_str].wv

        # Calculate metrics for the reference year itself (baseline)
        if y_str == ref_year_str:
            # For ref year, compare seed's neighbors with definition-based neighbors
            try:
                 ref_seed_neigh = [w for w, _ in ref_wv.most_similar(seed, topn=len(anchor_neighbors))] if seed in ref_wv else []
                 # Calculate overlap between definition neighbors and seed neighbors in ref year
                 overlap = len(set(anchor_neighbors) & set(ref_seed_neigh)) / max(1, len(anchor_neighbors))
            except Exception as e:
                 logging.warning(f"Could not calculate seed neighbors for overlap in ref year {ref_year_str}: {e}")
                 overlap = 0.0 # Assign 0 if error

            pos_change = 0.0
            sim = 1.0 # Similarity with itself
            # Fuzziness for ref year: Use overlap between definition neighbors and seed neighbors
            fuzziness = overlap # Use this specific overlap for ref year fuzziness
            category = categorize_membership(fuzziness)
            uncertainty = uncertainty_from_cat(category)
            # Log the Def/Seed overlap used for fuzziness here
            logging.info(f"  - W2V Year {y_str} (Reference): Positional Change = {pos_change:.3f}, Def/Seed Overlap (Fuzziness) = {overlap:.3f}")

        # Calculate metrics for subsequent years relative to reference
        else:
            # Find shared vocabulary for Procrustes alignment
            shared_for_align = intersect_vocab(ref_wv, tgt_wv, topn=min(topn_vocab, len(stable_anchor_words)))
            if len(shared_for_align) < min_align_words:
                logging.warning(f"[W2V {y_str}] Skipped: Too few shared words ({len(shared_for_align)} < {min_align_words}) with reference year for stable alignment.")
                continue

            # Compute alignment matrix R
            try:
                R = procrustes_R(ref_wv, tgt_wv, shared_for_align, dim)
            except Exception as e:
                logging.error(f"[W2V {y_str}] Procrustes alignment failed: {e}. Skipping year.")
                continue

            if seed not in tgt_wv.key_to_index:
                logging.warning(f"[W2V {y_str}] Seed word '{seed}' not found in vocabulary. Skipping year.")
                continue

            # Align the seed vector from the target year into the reference space
            seed_vec_aligned = np.dot(tgt_wv[seed], R)

            # Calculate metrics vs definition-based anchor_vec from reference year
            sim = 1 - cosine(anchor_vec, seed_vec_aligned) # Cosine Similarity
            pos_change = cosine(anchor_vec, seed_vec_aligned) # Cosine Distance

            # Neighborhood overlap: Compare target year seed neighbors with definition-based anchor_neighbors
            try:
                # Find neighbors of seed word *in the target year's original space*
                neigh_y_raw = [w for w, _ in tgt_wv.most_similar(seed, topn=len(anchor_neighbors))]
                overlap = len(set(anchor_neighbors) & set(neigh_y_raw)) / max(1, len(anchor_neighbors))
            except Exception as e:
                logging.warning(f"[W2V {y_str}] Could not compute neighbors or overlap for seed '{seed}': {e}")
                overlap = 0.0 # Assign 0 overlap if calculation fails

            fuzziness = overlap # Use overlap as proxy per spec for non-reference years
            category = categorize_membership(fuzziness)
            uncertainty = uncertainty_from_cat(category)

            # Log computed metrics for this year
            logging.info(f"  - W2V Year {y_str}: Positional Change = {pos_change:.3f}, Overlap (Fuzziness) = {overlap:.3f}")

        results.append({
            "year": y_str,
            "metrics": {
                "neighborhood_overlap": round(overlap, 3),
                "positional_change": round(pos_change, 3),
                "similarity_reduction": round(1 - sim, 3), # Retained for consistency
                "fuzziness_score": round(fuzziness, 3),
                "membership_category": category
            },
            "uncertainty": uncertainty
        })

    logging.info("Finished computing Word2Vec drift.")
    if not results:
        logging.warning("No W2V drift results were computed (check reference year and model availability).")
    return results


# --- Plotting Function ---
def plot_w2v_drift(seed, results, filename=None):
    """Plots Word2Vec drift metrics and saves if filename provided."""
    if not results:
        logging.warning("No W2V drift results to plot.")
        return
    try:
        years = [int(r["year"]) for r in results]
        pos_change = [r["metrics"]["positional_change"] for r in results]
        overlap = [r["metrics"]["neighborhood_overlap"] for r in results]
        fuzz = [r["metrics"]["fuzziness_score"] for r in results]

        fig, axs = plt.subplots(1, 3, figsize=(15, 5)) # Create figure and axes

        axs[0].plot(years, pos_change, marker='o')
        axs[0].set_title(f"Positional Change of '{seed}' (W2V)")
        axs[0].set_xlabel("Year"); axs[0].set_ylabel("Cosine Distance"); axs[0].grid(True)
        axs[0].set_ylim(bottom=0)

        axs[1].plot(years, overlap, marker='o')
        axs[1].set_title(f"Neighborhood Overlap of '{seed}' (W2V)")
        axs[1].set_xlabel("Year"); axs[1].set_ylabel("Overlap Ratio"); axs[1].grid(True)
        axs[1].set_ylim(0, 1.1)

        axs[2].plot(years, fuzz, marker='o')
        axs[2].set_title(f"Fuzziness Score of '{seed}' (W2V)")
        axs[2].set_xlabel("Year"); axs[2].set_ylabel("Fuzziness (Overlap)"); axs[2].grid(True)
        axs[2].set_ylim(0, 1.1)

        plt.tight_layout()

        if filename:
            try:
                plt.savefig(filename)
                logging.info(f"Saved W2V drift plot to {filename}")
            except Exception as e:
                logging.error(f"Failed to save W2V plot to {filename}: {e}")
        else:
            plt.show() # Display plot if not saving
        plt.close(fig) # Close the figure explicitly
    except Exception as e:
        logging.error(f"Failed to generate W2V drift plot: {e}")
        plt.close() # Ensure plot is closed even if error occurs

# --- JSON Output Function ---
def to_client_json(seed, definition, ref_year, anchor_neighbors, results, corpus_label="TXT+arXiv", model_name="Word2Vec", alignment_method="Orthogonal Procrustes"):
    """Creates the JSON structure for drift results."""
    return {
        "seed_word": seed,
        "context_anchor": {
            "definition": definition,
            "reference_year": ref_year,
            "anchor_neighbors": anchor_neighbors if anchor_neighbors is not None else []
        },
        "shift_moments": results if results else [],
        "provenance": {
            "embedding_model": model_name,
            "alignment_method": alignment_method,
            "corpus": corpus_label,
            "last_updated": datetime.now().date().isoformat()
        }
    }

def save_json(obj, path):
    """Saves a dictionary object to a JSON file."""
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        logging.info(f"Saved JSON data to: {path}")
    except Exception as e:
        logging.error(f"Failed to save JSON to {path}: {e}")


# ---- Run W2V drift end-to-end ----
logging.info("--- Starting W2V Drift Analysis Execution ---")
w2v_results = []
w2v_json_output = {}
ts = datetime.now().strftime("%Y%m%d-%H%M%S") # Timestamp for outputs

# Ensure models exist before proceeding
if 'w2v_models' in globals() and w2v_models:
    try:
        # Get anchor vector and DEFINITION-based neighbors
        anchor_vec_w2v, anchor_neigh_w2v = w2v_anchor_and_neighbors(
            w2v_models, REFERENCE_YEAR, SEED_WORD, ANCHOR_DEFINITION, TOPK_NEIGHBORS
        )
        logging.info(f"Definition-based Anchor Neighbors (W2V, Ref Year {REFERENCE_YEAR}): {anchor_neigh_w2v}")

        # Compute drift using the definition-based anchor vector and neighbors
        w2v_results = compute_w2v_drift(
            w2v_models, SEED_WORD, anchor_vec_w2v, anchor_neigh_w2v, REFERENCE_YEAR, topn_vocab=20000
        )

        # Generate Plot (Save to file - Ensure PLOT_DIR is defined in Cell 1)
        w2v_plot_filename = os.path.join(PLOT_DIR, f"semanticDrift_{SEED_WORD}_w2v_{ts}.png")
        plot_w2v_drift(SEED_WORD, w2v_results, filename=w2v_plot_filename)

        # Generate JSON structure
        w2v_json_output = to_client_json(
            SEED_WORD, ANCHOR_DEFINITION, REFERENCE_YEAR, anchor_neigh_w2v,
            w2v_results, "TXT+arXiv (merged)", "Word2Vec", "Orthogonal Procrustes"
            ) # Pass correct model name and method

        # (JSON saving moved to Cell 13 after backfill)

        logging.info("W2V Drift analysis and plotting complete.")
        # Use print for final summary if preferred over logging only
        print("\nWord2Vec JSON Structure Preview:")
        print({k: (type(v).__name__ if k != 'shift_moments' else f"{len(v)} moments") for k, v in w2v_json_output.items()})

    except ValueError as e:
        logging.error(f"Error during W2V drift analysis setup: {e}")
        # Use print for final error summary if preferred
        print(f"\nError during Word2Vec drift analysis: {e}")
        w2v_results = []
        w2v_json_output = {}
    except Exception as e:
        logging.error(f"An unexpected error occurred during W2V drift analysis: {e}", exc_info=True) # Log full traceback
        print(f"\nAn unexpected error occurred during Word2Vec drift analysis: {e}")
        w2v_results = []
        w2v_json_output = {}
else:
    logging.warning("Word2Vec models ('w2v_models') not found or empty. Skipping W2V drift analysis.")
    # Use print for final summary if preferred
    print("\nWord2Vec models were not trained successfully. Skipping drift analysis.")
    w2v_results = []
    w2v_json_output = {}


Word2Vec JSON Structure Preview:
{'seed_word': 'str', 'context_anchor': 'dict', 'shift_moments': '7 moments', 'provenance': 'dict'}


In [ ]:
# Cell 12: BERT Drift Analysis (Using Client's Sentence-Index Overlap)

!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer, util # Import util
from scipy.spatial.distance import cosine
import numpy as np
import matplotlib.pyplot as plt
import logging
import torch # Import torch
import re # Ensure re is imported

# --- BERT Centroid and Anchor Functions ---
def bert_year_centroid_for_seed(year_docs, seed, model_name="all-MiniLM-L6-v2", batch_size=64):
    """
    For each year, take docs that contain the seed term and compute the mean embedding.
    Returns dict: {year: vector}
    """
    model = SentenceTransformer(model_name)
    out = {}
    logging.info(f"Starting BERT centroid calculation using model: {model_name}")
    valid_years = sorted(year_docs.keys(), key=int)
    for y in valid_years:
        docs = year_docs[y]
        # Use regex for whole word match, case-insensitive
        subset = [d for d in docs if re.search(r'\b' + re.escape(seed.lower()) + r'\b', d.lower())]
        if not subset:
            logging.debug(f"BERT [{y}] No sentences found containing '{seed}'.")
            continue
        logging.info(f"BERT [{y}] Encoding {len(subset)} sentences containing '{seed}'...")
        try:
            embs = model.encode(subset, batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True)
            if embs.shape[0] > 0: # Check if embeddings were actually generated
                 out[y] = np.mean(embs, axis=0)
            else:
                 logging.warning(f"BERT [{y}] Encoding resulted in 0 embeddings.")
        except Exception as e:
            logging.error(f"Error encoding year {y} with BERT: {e}", exc_info=True)
    logging.info(f"Finished BERT centroid calculation. Centroids found for years: {list(out.keys())}")
    return out

def bert_definition_anchor(def_text, model_name="all-MiniLM-L6-v2"):
    """Encodes the anchor definition using BERT."""
    logging.info(f"Encoding anchor definition using BERT model: {model_name}")
    model = SentenceTransformer(model_name)
    try:
        return model.encode([def_text])[0]
    except Exception as e:
        logging.error(f"Error encoding anchor definition: {e}")
        raise

# --- CORRECTED compute_bert_drift function (using sentence-index overlap) ---
def compute_bert_drift(bert_year_vecs, anchor_vec, year_docs, seed, model_name="all-MiniLM-L6-v2", top_k_sentences=15):
    """
    Computes positional change (cosine distance vs anchor definition)
    and neighborhood overlap based on semantically closest *sentence indices* containing the seed word.
    Fuzziness is derived from this sentence-based overlap.
    """
    results = []
    logging.info("Computing BERT drift relative to anchor definition...")
    logging.info(f"Using Sentence-Index based Neighborhood Overlap (Top {top_k_sentences})")

    # --- Pre-compute sentence embeddings for overlap calculation ---
    model = SentenceTransformer(model_name)
    all_seed_sentences = []
    sentence_metadata = [] # Keep track of year and original index

    valid_years_for_overlap = sorted([y for y in year_docs.keys() if y in bert_year_vecs], key=int)
    logging.info(f"Years being considered for sentence overlap: {valid_years_for_overlap}")

    if not valid_years_for_overlap:
        logging.warning("No valid years found with BERT centroids for sentence overlap calculation. Overlap will be 0.")
        all_sentence_embeddings = None
    else:
        logging.info("Encoding all sentences containing the seed word for overlap calculation...")
        all_docs_list = []
        doc_counter = 0
        for year in valid_years_for_overlap:
             docs_in_year = year_docs.get(year, [])
             for doc in docs_in_year:
                  if re.search(r'\b' + re.escape(seed.lower()) + r'\b', doc.lower()):
                      all_docs_list.append(doc)
                      sentence_metadata.append({'year': year, 'original_doc_list_index': doc_counter})
                      doc_counter += 1

        if not all_docs_list:
             logging.warning(f"No sentences containing '{seed}' found in relevant years. Sentence overlap will be 0.")
             all_sentence_embeddings = None
        else:
             try:
                 all_sentence_embeddings = model.encode(all_docs_list, convert_to_tensor=True, show_progress_bar=True)
                 logging.info(f"Encoded {len(all_docs_list)} sentences for overlap calculation.")
             except Exception as e:
                 logging.error(f"Error encoding all sentences for overlap: {e}", exc_info=True)
                 all_sentence_embeddings = None

    # --- Calculate drift metrics year by year ---
    reference_sentence_indices = None # Store the indices from the reference year

    for idx, y_str in enumerate(valid_years_for_overlap):
        vec = bert_year_vecs[y_str]
        pos_change = cosine(np.array(anchor_vec), np.array(vec))
        cosine_sim = 1.0 - pos_change

        overlap = 0.0 # Default overlap

        # Calculate sentence neighborhood overlap if embeddings are available
        current_year_sentence_indices = set()
        if all_sentence_embeddings is not None and all_sentence_embeddings.shape[0] > 0:
             current_year_meta = [meta for meta in sentence_metadata if meta['year'] == y_str]
             if current_year_meta:
                 current_centroid_tensor = torch.tensor(vec, device=all_sentence_embeddings.device).unsqueeze(0) # Ensure 2D
                 indices_in_full_list = [meta['original_doc_list_index'] for meta in current_year_meta]

                 if indices_in_full_list:
                    similarities = util.cos_sim(current_centroid_tensor, all_sentence_embeddings)[0]
                    actual_k = min(top_k_sentences, len(similarities))
                    top_indices_all = torch.argsort(similarities, descending=True)[:actual_k]
                    current_year_sentence_indices = set(top_indices_all.cpu().tolist())

                    if idx == 0: # First year is reference
                        reference_sentence_indices = current_year_sentence_indices
                        overlap = 1.0
                        logging.info(f"  - BERT Year {y_str} (Overlap Reference): Top {len(reference_sentence_indices)} sentence indices stored.")
                    elif reference_sentence_indices is not None:
                        common_indices = reference_sentence_indices.intersection(current_year_sentence_indices)
                        overlap = len(common_indices) / max(1, actual_k)
                        logging.info(f"  - BERT Year {y_str}: Overlap with reference sentence indices: {overlap:.3f}")
                    else:
                         logging.warning(f"  - BERT Year {y_str}: Ref sentence indices missing.")
                 else:
                     logging.warning(f"  - BERT Year {y_str}: No sentences mapped.")
             else:
                  logging.warning(f"  - BERT Year {y_str}: No metadata found.")
        else:
             logging.warning(f"  - BERT Year {y_str}: Sentence embeddings NA, cannot calculate overlap.")

        fuzziness = overlap
        category = categorize_membership(fuzziness) if 'categorize_membership' in globals() else "n/a"
        uncertainty = uncertainty_from_cat(category) if 'uncertainty_from_cat' in globals() else "n/a"

        logging.info(f"  - BERT Year {y_str}: Positional Change = {pos_change:.3f}, Sentence Overlap (Fuzziness) = {fuzziness:.3f}")

        results.append({
            "year": y_str,
            "metrics": {
                "neighborhood_overlap": round(overlap, 3),
                "positional_change": round(pos_change, 3),
                "cosine_similarity": round(cosine_sim, 3),
                "fuzziness_score": round(fuzziness, 3),
                "membership_category": category
            },
            "uncertainty": uncertainty
        })

    logging.info("Finished computing BERT drift with sentence-based overlap.")
    return results

# --- Plotting Function (Modified to include overlap) ---
def plot_bert_drift(seed, results, filename=None): # Added filename argument
    """ Plots BERT drift metrics and saves if filename provided. """
    if not results:
        logging.warning("No BERT drift results to plot.")
        return
    try:
        years = [int(r["year"]) for r in results]
        pos_change = [r["metrics"].get("positional_change", 0) for r in results]
        cosine_sim = [r["metrics"].get("cosine_similarity", 0) for r in results]
        overlap = [r["metrics"].get("neighborhood_overlap", 0) for r in results] # Get overlap data

        fig, axs = plt.subplots(1, 2, figsize=(12, 5)) # 2 subplots

        # Plot 1: Positional Change / Similarity
        axs[0].plot(years, pos_change, marker='o', label="Positional Change (Distance)")
        axs[0].plot(years, cosine_sim, marker='s', label="Cosine Similarity")
        axs[0].set_title(f"BERT vs Definition Anchor for '{seed}'")
        axs[0].set_xlabel("Year"); axs[0].set_ylabel("Cosine Score")
        axs[0].set_ylim(0, max(1.1, max(pos_change)*1.1 if pos_change else 1.1))
        axs[0].grid(True); axs[0].legend()

         # Plot 2: Sentence Overlap
        axs[1].plot(years, overlap, marker='^', label="Sentence Overlap (Fuzziness)", color='green')
        axs[1].set_title(f"Sentence Neighborhood Overlap for '{seed}' (BERT)")
        axs[1].set_xlabel("Year"); axs[1].set_ylabel("Overlap Ratio")
        axs[1].set_ylim(0, 1.1)
        axs[1].grid(True); axs[1].legend()

        plt.tight_layout()

        if filename:
             try:
                 plt.savefig(filename)
                 logging.info(f"Saved BERT drift plot to {filename}")
             except Exception as e:
                 logging.error(f"Failed to save BERT plot to {filename}: {e}")
        else:
             plt.show()
        plt.close(fig) # Close the figure

    except Exception as e:
        logging.error(f"Failed to generate BERT drift plot: {e}", exc_info=True)
        plt.close()


# ---- Run BERT drift ----
logging.info("--- Starting BERT Drift Analysis ---")
bert_results = [] # Initialize
try:
    if 'ANCHOR_DEFINITION' not in globals() or 'SEED_WORD' not in globals() or 'year_docs' not in globals():
         raise NameError("Required variables (ANCHOR_DEFINITION, SEED_WORD, year_docs) not defined.")

    bert_anchor = bert_definition_anchor(ANCHOR_DEFINITION, model_name=BERT_MODEL_NAME)
    bert_year_vecs = bert_year_centroid_for_seed(year_docs, SEED_WORD, model_name=BERT_MODEL_NAME, batch_size=BERT_BATCH_SIZE)

    if bert_year_vecs: # Only proceed if centroids were computed
        bert_results = compute_bert_drift(
            bert_year_vecs,
            bert_anchor,
            year_docs,            # Pass year_docs
            SEED_WORD,            # Pass seed word
            model_name=BERT_MODEL_NAME,
            top_k_sentences=TOPK_NEIGHBORS # Use param from Cell 1
        )
    else:
        logging.warning("BERT centroids could not be computed. Skipping drift calculation.")
        bert_results = []

    ts_plot = datetime.now().strftime("%Y%m%d-%H%M%S")
    bert_plot_filename = os.path.join(PLOT_DIR, f"semanticDrift_{SEED_WORD}_bert_{ts_plot}.png")
    plot_bert_drift(SEED_WORD, bert_results, filename=bert_plot_filename) # Pass filename

except NameError as e:
    logging.error(f"Cannot run BERT analysis: {e}")
    bert_results = []
except Exception as e:
    logging.error(f"An unexpected error occurred during BERT drift analysis: {e}", exc_info=True)
    bert_results = []

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# Cell 13: Save Initial Drift JSONs (Corrected)

import os, json
from datetime import datetime
import numpy as np # Import numpy to handle float32/64 conversion

# Define numpy-safe save function
def save_json_with_numpy(obj, path):
    """Saves a Python object to JSON, handling numpy types."""
    def default_converter(o):
        if isinstance(o, (np.float32, np.float64)): return float(o)
        if isinstance(o, (np.int32, np.int64)): return int(o)
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, (datetime, pd.Timestamp)): return o.isoformat()
        raise TypeError(f"Object of type {o.__class__.__name__} is not JSON serializable")
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2, default=default_converter)
        logging.info(f"Saved JSON with numpy conversion: {path}")
    except Exception as e:
        logging.error(f"Error saving {path} with numpy conversion: {e}", exc_info=True)

# --- Use DRIFT_JSON_DIR defined in Cell 1 ---
OUT_DIR = DRIFT_JSON_DIR
os.makedirs(OUT_DIR, exist_ok=True)
logging.info(f"Saving initial drift JSONs to: {OUT_DIR}")

# Timestamp + seed helpers
ts_save = datetime.now().strftime("%Y%m%d-%H%M%S")
seed = SEED_WORD if 'SEED_WORD' in globals() else "seed"

# --- Save W2V results ('w2v_json_output' from Cell 10) ---
w2v_path = None # Initialize path
if 'w2v_json_output' in globals() and w2v_json_output:
    w2v_path = os.path.join(OUT_DIR, f"semanticDrift_{seed}_w2v_{ts_save}.json")
    save_json_with_numpy(w2v_json_output, w2v_path)
else:
    logging.warning("'w2v_json_output' not found or empty. W2V JSON not saved.")

# --- Save BERT results ('bert_results' from Cell 11) ---
bert_path = None # Initialize path
if 'bert_results' in globals() and bert_results:
    ref_neighbors_for_bert = anchor_neigh_w2v if 'anchor_neigh_w2v' in globals() else []
    if not ref_neighbors_for_bert:
         logging.warning("W2V anchor neighbors ('anchor_neigh_w2v') not found. Using empty list for BERT JSON.")

    bert_ref_year = min(bert_year_vecs.keys()) if 'bert_year_vecs' in globals() and bert_year_vecs else REFERENCE_YEAR

    # Use the to_client_json function (defined in Cell 10)
    bert_json_to_save = to_client_json(
        SEED_WORD, ANCHOR_DEFINITION, bert_ref_year,
        ref_neighbors_for_bert, # Use W2V def-based neighbors
        bert_results, "TXT+arXiv (merged)", BERT_MODEL_NAME, "Centroid vs Definition + Sentence Overlap"
     )

    bert_path = os.path.join(OUT_DIR, f"semanticDrift_{SEED_WORD}_bert_{ts_save}.json")
    save_json_with_numpy(bert_json_to_save, bert_path)
else:
    logging.warning("'bert_results' not found or empty. BERT JSON not saved.")

# Quick check of the directory
print(f"\n--- Contents of {OUT_DIR} ---")
!ls -lht "{OUT_DIR}" | head -n 10


--- Contents of /content/drive/MyDrive/Projects/Upwork/outputs/drift_json ---
total 130K
-rw------- 1 root root 4.5K Nov  1 07:58 semanticDrift_cloud_bert_20251101-075811.json
-rw------- 1 root root 2.9K Nov  1 07:58 semanticDrift_cloud_w2v_20251101-075811.json
-rw------- 1 root root  471 Oct 25 16:19 semanticChangeReport_cloud_20251025-161921.json
-rw------- 1 root root  471 Oct 25 16:09 semanticChangeReport_cloud_20251025-160950.json
-rw------- 1 root root 4.5K Oct 25 16:07 semanticDrift_cloud_bert_20251025-160748.json
-rw------- 1 root root 2.9K Oct 25 16:07 semanticDrift_cloud_w2v_20251025-160748.json
-rw------- 1 root root  541 Oct 25 13:08 semanticChangeReport_cloud_20251025-130801.json
-rw------- 1 root root 4.3K Oct 25 13:03 semanticDrift_cloud_bert_20251025-130314.json
-rw------- 1 root root 2.9K Oct 25 13:03 semanticDrift_cloud_w2v_20251025-130314.json


In [ ]:
# Cell 14: Backfill Membership Scores in Saved JSONs

import os, json
from pathlib import Path
import numpy as np
from datetime import datetime

# --- Membership Helper Functions (already defined in Cell 9) ---

# --- Backfill Function ---
def backfill_membership(json_obj):
    """
    Adds/repairs fuzziness, membership, and uncertainty IN-PLACE.
    Uses 'neighborhood_overlap' if available, otherwise calculates from 'similarity_reduction' (W2V) or 'cosine_similarity' (BERT).
    """
    if not json_obj or "shift_moments" not in json_obj: return 0

    n_moments_processed = 0
    is_bert = "bert" in json_obj.get("provenance", {}).get("embedding_model", "").lower()

    for moment in json_obj["shift_moments"]:
        metrics = moment.setdefault("metrics", {})
        fuzz = metrics.get("fuzziness_score")
        overlap = metrics.get("neighborhood_overlap")
        sim_red = metrics.get("similarity_reduction") # W2V specific
        cos_sim = metrics.get("cosine_similarity") # BERT specific

        # Determine fuzziness score
        if fuzz is None or np.isnan(fuzz):
             if overlap is not None and not np.isnan(overlap):
                 fuzz = clip01(overlap)
             elif sim_red is not None and not np.isnan(sim_red) and not is_bert:
                 fuzz = clip01(1.0 - sim_red) # Fuzziness = 1 - distance
             elif cos_sim is not None and not np.isnan(cos_sim) and is_bert:
                 fuzz = clip01(cos_sim) # Fuzziness = similarity
             else:
                 fuzz = 0.0
             metrics["fuzziness_score"] = round(fuzz, 3)
        else:
             if np.isnan(fuzz): fuzz = 0.0
             metrics["fuzziness_score"] = round(clip01(fuzz), 3)

        category = categorize_membership(metrics["fuzziness_score"])
        uncertainty = uncertainty_from_cat(category)
        metrics["membership_category"] = category
        moment["uncertainty"] = uncertainty

        n_moments_processed += 1

    if "provenance" in json_obj:
         json_obj["provenance"]["last_updated"] = datetime.now().date().isoformat() + " (backfilled)"

    return n_moments_processed

# --- Load, Backfill, and Save ---
logging.info("Starting backfill process for membership scores...")

# Use the paths saved in the previous cell
paths_to_process = [(w2v_path, "W2V"), (bert_path, "BERT")]

for file_path, label in paths_to_process:
    if file_path and Path(file_path).exists():
        logging.info(f"Processing {label} file: {file_path}")
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            count = backfill_membership(data)
            save_json_with_numpy(data, file_path) # Use numpy-safe save
            logging.info(f" Backfilled and saved {count} moments for {label}.")
        except Exception as e:
            logging.error(f" Error processing {file_path}: {e}", exc_info=True)
    else:
        logging.warning(f"{label} JSON path ('{file_path}') not found. Skipping backfill.")

logging.info("Backfill process complete.")

# --- Quick Preview ---
def preview_backfilled(path, label):
    if not path or not Path(path).exists(): return
    try:
        with open(path, "r", encoding="utf-8") as f: obj = json.load(f)
        sms = obj.get("shift_moments", [])
        print(f"\n{label} Preview After Backfill ({len(sms)} moments):")
        for m in sms[:3]:
            y = m.get("year", "N/A")
            met = m.get("metrics", {})
            fuzz = met.get('fuzziness_score', 'N/A')
            cat = met.get('membership_category', 'N/A')
            unc = m.get('uncertainty', 'N/A')
            print(f"  {y} -> Fuzz={fuzz}, Cat={cat}, Unc={unc}")
    except Exception as e: print(f"{label} preview failed: {e}")

preview_backfilled(w2v_path, "W2V")
preview_backfilled(bert_path, "BERT")

ERROR:root: Error processing /content/drive/MyDrive/Projects/Upwork/outputs/drift_json/semanticDrift_cloud_w2v_20251101-075811.json: name 'clip01' is not defined
Traceback (most recent call last):
  File "/tmp/ipython-input-352565398.py", line 67, in <cell line: 0>
    count = backfill_membership(data)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-352565398.py", line 41, in backfill_membership
    metrics["fuzziness_score"] = round(clip01(fuzz), 3)
                                       ^^^^^^
NameError: name 'clip01' is not defined
ERROR:root: Error processing /content/drive/MyDrive/Projects/Upwork/outputs/drift_json/semanticDrift_cloud_bert_20251101-075811.json: name 'clip01' is not defined
Traceback (most recent call last):
  File "/tmp/ipython-input-352565398.py", line 67, in <cell line: 0>
    count = backfill_membership(data)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-352565398.py", line 41, in backfill_membership
    metrics["fuzziness_sc


W2V Preview After Backfill (7 moments):
  2007 -> Fuzz=0.0, Cat=unstable, Unc=provisional
  2008 -> Fuzz=0.0, Cat=unstable, Unc=provisional
  2009 -> Fuzz=0.0, Cat=unstable, Unc=provisional

BERT Preview After Backfill (12 moments):
  2007 -> Fuzz=1.0, Cat=stable, Unc=low
  2008 -> Fuzz=0.533, Cat=transitional, Unc=medium
  2009 -> Fuzz=0.667, Cat=transitional, Unc=medium


In [ ]:
# Cell 16: VALIDATION - Step 1: Define analysis span
# (Uses SHIFT_YEAR_1 and SHIFT_YEAR_2 from Cell 1)

anchor_word_input = SEED_WORD
shift_year_1_input = SHIFT_YEAR_1
shift_year_2_input = SHIFT_YEAR_2

analysis_span = tuple(sorted([shift_year_1_input, shift_year_2_input]))

print(f"Anchor word for validation analysis: {anchor_word_input}")
print(f"Shift years for validation analysis: {shift_year_1_input}, {shift_year_2_input}")
print(f"Defined validation analysis span: {analysis_span}")

Anchor word for validation analysis: cloud
Shift years for validation analysis: 2008, 2011
Defined validation analysis span: (2008, 2011)


In [ ]:
# Cell 17: VALIDATION - Step 2: Extract and Preprocess Sentences

# Step 1: Compute midpoint
start_year, end_year = analysis_span
midpoint_year = (start_year + end_year) / 2

# Step 2: Define symmetric time window (uses VAL_WINDOW_HALF_WIDTH from Cell 1)
half_window_width = VAL_WINDOW_HALF_WIDTH
window_start_year = int(midpoint_year - half_window_width)
window_end_year = int(midpoint_year + half_window_width)

print(f"Midpoint year: {midpoint_year}")
print(f"Symmetric window for validation: {window_start_year} - {window_end_year}")

# Step 3 & 4: Extract sentences within the window
extracted_sentences = []
anchor_word = SEED_WORD

if 'year_docs' not in globals():
    raise NameError("Variable 'year_docs' not found. Run Cell 5 first.")

for year in sorted(year_docs.keys(), key=lambda s: int(s)):
    year_int = int(year)
    if window_start_year <= year_int <= window_end_year:
        if year in year_docs:
            for sentence in year_docs[year]:
                if re.search(r'\b' + re.escape(anchor_word.lower()) + r'\b', sentence.lower()):
                    extracted_sentences.append({
                        "year": year_int,
                        "corpus": "Merged (TXT+arXiv)",
                        "raw_sentence": sentence # Store original
                    })

print(f"Extracted {len(extracted_sentences)} sentences containing '{anchor_word}' within {window_start_year}-{window_end_year}.")

# Step 5: Preprocess sentences FOR BERTOPIC ONLY (MLM uses raw)
preprocessed_sentences_topic = []
for sentence_info in extracted_sentences:
    cleaned_tokens = simple_clean(sentence_info["raw_sentence"])
    cleaned_sentence = " ".join(cleaned_tokens)
    if cleaned_sentence: # Only add if not empty
        preprocessed_sentences_topic.append({
            "year": sentence_info["year"],
            "corpus": sentence_info["corpus"],
            "processed_sentence": cleaned_sentence
        })

print(f"Preprocessed {len(preprocessed_sentences_topic)} non-empty sentences for BERTopic.")

Midpoint year: 2009.5
Symmetric window for validation: 2007 - 2011
Extracted 345 sentences containing 'cloud' within 2007-2011.
Preprocessed 345 non-empty sentences for BERTopic.


In [ ]:
# Cell 18: VALIDATION - Step 3: Define Metric Functions
# (Originally Cell 23)

import numpy as np
from scipy.spatial.distance import jensenshannon
import json
import logging
from datetime import datetime
import pandas as pd

# (Functions are defined in Cell 13 now, but we define JSD/RankOverlap again for this section)
def jensen_shannon_divergence(dist1, dist2):
    """Computes the Jensen-Shannon divergence between two probability distributions (dictionaries)."""
    all_tokens = set(dist1.keys()) | set(dist2.keys())
    if not all_tokens: return 0.0

    p1 = np.array([dist1.get(token, 0.0) for token in all_tokens])
    p2 = np.array([dist2.get(token, 0.0) for token in all_tokens])

    p1_sum = np.sum(p1); p2_sum = np.sum(p2)
    if p1_sum <= 0: p1 = np.ones_like(p1) / len(p1) if len(p1) > 0 else np.array([1.0])
    else: p1 /= p1_sum
    if p2_sum <= 0: p2 = np.ones_like(p2) / len(p2) if len(p2) > 0 else np.array([1.0])
    else: p2 /= p2_sum

    if len(p1) != len(p2):
        logging.error(f"Length mismatch in JSD vectors: {len(p1)} vs {len(p2)}")
        return 1.0 # Max divergence

    epsilon = 1e-10
    p1 = np.maximum(p1, epsilon); p2 = np.maximum(p2, epsilon)
    p1 /= np.sum(p1); p2 /= np.sum(p2)

    try:
      js_div_sqrt = jensenshannon(p1, p2, base=2)
      return js_div_sqrt**2 if not np.isnan(js_div_sqrt) else 1.0
    except ValueError as e:
        logging.error(f"JSD Error: {e}. p1 sum: {np.sum(p1)}, p2 sum: {np.sum(p2)}")
        return 1.0

def rank_overlap(dist1, dist2, top_k):
    """Computes the rank overlap between the top_k tokens of two distributions (dictionaries)."""
    if top_k <= 0: return 0.0
    if not dist1 or not dist2: return 0.0
    top_tokens1 = sorted(dist1, key=dist1.get, reverse=True)[:top_k]
    top_tokens2 = sorted(dist2, key=dist2.get, reverse=True)[:top_k]
    common_tokens = set(top_tokens1) & set(top_tokens2)
    return len(common_tokens) / max(1, top_k)

print("Validation metric functions `jensen_shannon_divergence` and `rank_overlap` defined.")

Validation metric functions `jensen_shannon_divergence` and `rank_overlap` defined.


In [ ]:
# Cell 19: VALIDATION - Step 4: Run MLM
# (Originally Cell 19)

try:
    from transformers import pipeline, logging as hf_logging
    hf_logging.set_verbosity_error()
except ImportError:
    !pip install -q transformers torch
    from transformers import pipeline, logging as hf_logging
    hf_logging.set_verbosity_error()

import torch
from collections import defaultdict
import random
import nltk
from nltk.corpus import stopwords
import logging

try: stopwords.words('english')
except LookupError: nltk.download('stopwords')

# Helper function
def clean_mlm_predictions(predictions, anchor_word):
    stop_words = set(stopwords.words('english'))
    cleaned = []
    anchor_plural = anchor_word + 's'
    for pred in predictions:
        token = pred['token_str'].strip().lower()
        if (token in [anchor_word, anchor_plural] or
            token in stop_words or
            not token.isalpha() or
            len(token) < 2 or
            token.startswith('##') or
            token in ['[unk]', '[cls]', '[sep]', '[pad]', '[mask]']):
            continue
        cleaned.append(pred)
    return cleaned

# Parameters from Cell 1
mlm_model_name = MLM_MODEL_NAME
num_sentences_per_year = MLM_SENTENCES_PER_YEAR

# Initialize model
try:
    if 'unmasker' not in globals() or unmasker is None:
        unmasker = pipeline("fill-mask", model=mlm_model_name, device=0 if torch.cuda.is_available() else -1)
        logging.info(f"Initialized MLM model: {mlm_model_name} on device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
    else:
         logging.info(f"Using existing MLM model: {mlm_model_name}")
except Exception as e:
    logging.error(f"Error initializing MLM model {mlm_model_name}: {e}")
    unmasker = None

yearly_mlm_distributions = {}
if unmasker:
    anchor_word_lower = SEED_WORD.lower()
    if 'extracted_sentences' not in globals():
        logging.error("Variable 'extracted_sentences' not found for MLM.")
    else:
        sentences_by_year_raw = defaultdict(list)
        for sent_info in extracted_sentences:
             if analysis_span[0] <= sent_info["year"] <= analysis_span[1]:
                sentences_by_year_raw[sent_info["year"]].append(sent_info["raw_sentence"])

        logging.info(f"Starting MLM prediction for validation span years {analysis_span[0]} to {analysis_span[1]}...")
        years_to_process = sorted([y for y in sentences_by_year_raw.keys()])

        for year in years_to_process:
            year_sentences = sentences_by_year_raw[year]
            logging.info(f"MLM [{year}] Processing {len(year_sentences)} sentences...")
            if not year_sentences:
                yearly_mlm_distributions[year] = {}
                continue

            k_sample = min(num_sentences_per_year, len(year_sentences))
            sampled_sentences = random.sample(year_sentences, k_sample)
            logging.info(f"  Sampled {k_sample} sentences.")

            yearly_predictions = defaultdict(float)
            prediction_count = 0
            for sentence in sampled_sentences:
                tokens = sentence.split()
                masked_sentence_tokens = []
                anchor_indices = [i for i, token in enumerate(tokens) if token.lower() == anchor_word_lower]
                if not anchor_indices: continue
                first_anchor_index = anchor_indices[0]
                masked_sentence_tokens = tokens[:first_anchor_index] + [unmasker.tokenizer.mask_token] + tokens[first_anchor_index+1:]
                masked_sentence = " ".join(masked_sentence_tokens)

                try:
                    preds = unmasker(masked_sentence, top_k=50)
                    cleaned_preds = clean_mlm_predictions(preds, anchor_word_lower)
                    if cleaned_preds:
                        for pred in cleaned_preds: yearly_predictions[pred['token_str']] += pred['score']
                        prediction_count += 1
                except Exception as e:
                    logging.warning(f"  MLM Error on sentence: '{masked_sentence[:50]}...'. Error: {e}")

            total_score = sum(yearly_predictions.values())
            if total_score > 0 and prediction_count > 0:
                yearly_mlm_distributions[year] = {t: s / total_score for t, s in yearly_predictions.items()}
                logging.info(f"  Aggregated predictions for {prediction_count} sentences.")
            else:
                 logging.warning(f"  No valid MLM predictions for year {year}.")
                 yearly_mlm_distributions[year] = {}

    logging.info("Finished MLM prediction for validation.")
    print("\n--- Validation Yearly MLM Distributions (Preview) ---")
    for year in sorted(yearly_mlm_distributions.keys())[:3]: # Show first 3
        dist = yearly_mlm_distributions[year]
        print(f"Year {year}:")
        top_tokens = sorted(dist.items(), key=lambda item: item[1], reverse=True)[:5]
        if top_tokens:
            for token, prob in top_tokens: print(f"  {token}: {prob:.4f}")
        else: print("  (No valid data)")
else:
    print("MLM model ('unmasker') could not be initialized. Skipping validation MLM.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


--- Validation Yearly MLM Distributions (Preview) ---
Year 2008:
  large: 0.0403
  particle: 0.0329
  snow: 0.0309
  field: 0.0267
  magnetic: 0.0263
Year 2009:
  star: 0.0729
  phase: 0.0572
  maximum: 0.0432
  scale: 0.0404
  gravitational: 0.0312
Year 2010:
  dust: 0.3176
  gas: 0.1341
  light: 0.0833
  objects: 0.0250
  clusters: 0.0137


In [ ]:
# Cell 20: VALIDATION - Step 5: Calculate Adjacent Year MLM Metrics
# (Originally Cell 24)

adjacent_year_metrics = []
if 'yearly_mlm_distributions' not in globals() or not yearly_mlm_distributions:
    logging.error("Validation MLM distributions not found.")
else:
    years_with_data = sorted([y for y in yearly_mlm_distributions.keys() if analysis_span[0] <= y <= analysis_span[1]], key=int)
    print(f"Calculating adjacent year MLM metrics for: {years_with_data}")

    for i in range(len(years_with_data) - 1):
        year1, year2 = years_with_data[i], years_with_data[i+1]
        dist1, dist2 = yearly_mlm_distributions.get(year1), yearly_mlm_distributions.get(year2)

        if not dist1 or not dist2:
            logging.warning(f"  Skipping {year1}-{year2}: empty distribution.")
            jsd_val, overlap_val = None, None
        else:
            try:
                jsd_val = jensen_shannon_divergence(dist1, dist2)
                overlap_val = rank_overlap(dist1, dist2, MLM_RANK_OVERLAP_K)
                print(f"  Metrics {year1}-{year2}: JSD={jsd_val:.4f}, Overlap@{MLM_RANK_OVERLAP_K}={overlap_val:.4f}")
            except Exception as e:
                 logging.error(f"Error metrics {year1}-{year2}: {e}")
                 jsd_val, overlap_val = None, None

        adjacent_year_metrics.append({
            "year_pair": f"{year1}-{year2}",
            "jensen_shannon_divergence": round(jsd_val, 4) if jsd_val is not None else None,
            "rank_overlap": round(overlap_val, 4) if overlap_val is not None else None
        })

print("\n--- Adjacent Year MLM Metrics Summary ---")
if adjacent_year_metrics:
    try:
        import pandas as pd
        display(pd.DataFrame(adjacent_year_metrics))
    except ImportError:
        for metrics in adjacent_year_metrics: print(metrics)
else:
    print("No adjacent MLM metrics calculated.")

Calculating adjacent year MLM metrics for: [2008, 2009, 2010, 2011]
  Metrics 2008-2009: JSD=0.6723, Overlap@10=0.0000
  Metrics 2009-2010: JSD=0.6414, Overlap@10=0.1000
  Metrics 2010-2011: JSD=0.7295, Overlap@10=0.2000

--- Adjacent Year MLM Metrics Summary ---


,year_pair,jensen_shannon_divergence,rank_overlap
0,2008-2009,0.6723,0.0
1,2009-2010,0.6414,0.1
2,2010-2011,0.7295,0.2


In [ ]:
# Cell 21: VALIDATION - Step 6: Bootstrap Resampling for MLM Metrics
# (Originally Cell 20)

from collections import defaultdict
import random
import numpy as np
import logging

def bootstrap_mlm_metrics(sentences_by_year_raw, unmasker, anchor_word, num_sentences_per_year, analysis_span, num_bootstrap_samples=50, top_k_overlap=10):
    bootstrap_results = defaultdict(lambda: defaultdict(list))
    years_in_span_with_data = sorted([y for y in sentences_by_year_raw.keys() if analysis_span[0] <= y <= analysis_span[1]], key=int)

    if len(years_in_span_with_data) < 2:
         logging.warning("Need at least two years with data in span for bootstrap comparisons.")
         return {}

    logging.info(f"Bootstrap MLM ({num_bootstrap_samples} samples) on years: {years_in_span_with_data}")

    for i in range(num_bootstrap_samples):
        if (i + 1) % 10 == 0: logging.info(f"  Bootstrap sample {i+1}/{num_bootstrap_samples}...")
        bootstrap_yearly_distributions = {}

        for year in years_in_span_with_data:
            year_sentences = sentences_by_year_raw[year]
            if not year_sentences: continue

            k_sample_size = min(num_sentences_per_year, len(year_sentences))
            if k_sample_size <=0: continue
            sampled_sentences = random.choices(year_sentences, k=k_sample_size)

            yearly_predictions = defaultdict(float)
            prediction_count = 0
            for sentence in sampled_sentences:
                tokens = sentence.split()
                anchor_indices = [idx for idx, token in enumerate(tokens) if token.lower() == anchor_word.lower()]
                if not anchor_indices: continue
                first_anchor_index = anchor_indices[0]
                masked_sentence_tokens = tokens[:first_anchor_index] + [unmasker.tokenizer.mask_token] + tokens[first_anchor_index+1:]
                masked_sentence = " ".join(masked_sentence_tokens)

                try:
                    preds = unmasker(masked_sentence, top_k=50)
                    cleaned_preds = clean_mlm_predictions(preds, anchor_word.lower())
                    if cleaned_preds:
                        for pred in cleaned_preds: yearly_predictions[pred['token_str']] += pred['score']
                        prediction_count += 1
                except Exception: pass

            total_score = sum(yearly_predictions.values())
            if total_score > 0 and prediction_count > 0:
                bootstrap_yearly_distributions[year] = {t: s / total_score for t, s in yearly_predictions.items()}
            else: bootstrap_yearly_distributions[year] = {}

        for j in range(len(years_in_span_with_data) - 1):
            year1, year2 = years_in_span_with_data[j], years_in_span_with_data[j+1]
            year_pair = f"{year1}-{year2}"
            dist1, dist2 = bootstrap_yearly_distributions.get(year1), bootstrap_yearly_distributions.get(year2)

            if dist1 and dist2:
                try:
                    jsd = jensen_shannon_divergence(dist1, dist2)
                    overlap = rank_overlap(dist1, dist2, top_k_overlap)
                    bootstrap_results[year_pair]["jsd"].append(jsd)
                    bootstrap_results[year_pair]["rank_overlap"].append(overlap)
                except Exception: pass

    logging.info("Bootstrap resampling complete.")
    for pair, data in bootstrap_results.items():
        logging.info(f"  {pair}: Got {len(data.get('jsd', []))} JSD, {len(data.get('rank_overlap', []))} Overlap samples.")

    return dict(bootstrap_results)

# --- Run Bootstrap ---
sentences_by_year_raw = defaultdict(list)
for sent_info in extracted_sentences: # Use extracted_sentences with raw data
    if analysis_span[0] <= sent_info["year"] <= analysis_span[1]:
        sentences_by_year_raw[sent_info["year"]].append(sent_info["raw_sentence"])

bootstrap_mlm_results = {}
if unmasker and sentences_by_year_raw:
    bootstrap_mlm_results = bootstrap_mlm_metrics(
        sentences_by_year_raw, unmasker, SEED_WORD,
        MLM_SENTENCES_PER_YEAR, analysis_span,
        num_bootstrap_samples=MLM_BOOTSTRAP_SAMPLES,
        top_k_overlap=MLM_RANK_OVERLAP_K
    )
elif not unmasker: logging.error("MLM unmasker NA, skipping bootstrap.")
else: logging.warning("No sentences for MLM bootstrap.")

# Print Sample Results
print("\n--- Bootstrap MLM Results (Sample Preview) ---")
if bootstrap_mlm_results:
    for year_pair, metrics_data in list(bootstrap_mlm_results.items())[:3]:
        print(f"Year Pair {year_pair}:")
        jsd_samples = metrics_data.get('jsd', [])
        overlap_samples = metrics_data.get('rank_overlap', [])
        print(f"  JSD samples ({len(jsd_samples)}): {jsd_samples[:5]}...")
        print(f"  Overlap samples ({len(overlap_samples)}): {overlap_samples[:5]}...")
else:
    print("No bootstrap results generated.")


--- Bootstrap MLM Results (Sample Preview) ---
Year Pair 2008-2009:
  JSD samples (50): [np.float64(0.7239838504000329), np.float64(0.6093520115088183), np.float64(0.6991365997953686), np.float64(0.6383291940972127), np.float64(0.7562304808988641)]...
  Overlap samples (50): [0.1, 0.1, 0.0, 0.0, 0.1]...
Year Pair 2009-2010:
  JSD samples (50): [np.float64(0.4407781097240423), np.float64(0.6159109261048408), np.float64(0.6911635876975541), np.float64(0.816392092763315), np.float64(0.5919074868169266)]...
  Overlap samples (50): [0.4, 0.2, 0.2, 0.0, 0.2]...
Year Pair 2010-2011:
  JSD samples (50): [np.float64(0.7142852212750505), np.float64(0.7919938766917509), np.float64(0.8312763149959048), np.float64(0.908507867055583), np.float64(0.8925455725569428)]...
  Overlap samples (50): [0.1, 0.0, 0.0, 0.0, 0.0]...


In [ ]:
# Cell 22: VALIDATION - Step 7: Calculate Confidence Intervals
# (Originally Cell 31)

import numpy as np

def calculate_confidence_intervals(bootstrap_results, confidence_level=0.95):
    """Calculates percentile-based confidence intervals from bootstrap results."""
    confidence_intervals = {}
    if not bootstrap_results:
        logging.warning("Bootstrap results empty. Cannot calculate CIs.")
        return confidence_intervals

    alpha = 1.0 - confidence_level
    lower_percentile = alpha / 2.0 * 100
    upper_percentile = (1.0 - alpha / 2.0) * 100

    logging.info(f"Calculating {confidence_level*100:.0f}% confidence intervals...")

    for year_pair, metrics_data in bootstrap_results.items():
        year_pair_cis = {}

        jsd_samples = metrics_data.get("jsd", [])
        if jsd_samples:
            try:
                jsd_samples_float = [float(s) for s in jsd_samples if s is not None and not np.isnan(s)]
                if jsd_samples_float:
                    jsd_ci_lower = np.percentile(jsd_samples_float, lower_percentile)
                    jsd_ci_upper = np.percentile(jsd_samples_float, upper_percentile)
                    year_pair_cis["jensen_shannon_divergence_ci"] = (round(jsd_ci_lower, 4), round(jsd_ci_upper, 4))
                    logging.info(f"  {year_pair} JSD CI: ({jsd_ci_lower:.4f}, {jsd_ci_upper:.4f})")
                else: logging.warning(f"  No valid JSD samples for {year_pair}.")
            except Exception as e: logging.error(f"  Error JSD CI for {year_pair}: {e}")
        else: logging.warning(f"  No JSD samples for {year_pair}.")

        overlap_samples = metrics_data.get("rank_overlap", [])
        if overlap_samples:
             try:
                overlap_samples_float = [float(s) for s in overlap_samples if s is not None and not np.isnan(s)]
                if overlap_samples_float:
                    overlap_ci_lower = np.percentile(overlap_samples_float, lower_percentile)
                    overlap_ci_upper = np.percentile(overlap_samples_float, upper_percentile)
                    year_pair_cis["rank_overlap_ci"] = (round(overlap_ci_lower, 4), round(overlap_ci_upper, 4))
                    logging.info(f"  {year_pair} Overlap CI: ({overlap_ci_lower:.4f}, {overlap_ci_upper:.4f})")
                else: logging.warning(f"  No valid Overlap samples for {year_pair}.")
             except Exception as e: logging.error(f"  Error Overlap CI for {year_pair}: {e}")
        else: logging.warning(f"  No Rank Overlap samples for {year_pair}.")

        if year_pair_cis:
            confidence_intervals[year_pair] = year_pair_cis

    return confidence_intervals

mlm_confidence_intervals = calculate_confidence_intervals(bootstrap_mlm_results, confidence_level=0.95)

print("\n--- MLM Confidence Intervals Summary ---")
if mlm_confidence_intervals:
    try:
        import pandas as pd
        ci_list = [{'year_pair': k, **v} for k, v in mlm_confidence_intervals.items()]
        display(pd.DataFrame(ci_list).set_index('year_pair'))
    except ImportError:
        for year_pair, cis in mlm_confidence_intervals.items(): print(f"{year_pair}: {cis}")
else:
    print("No MLM confidence intervals were calculated.")


--- MLM Confidence Intervals Summary ---


,jensen_shannon_divergence_ci,rank_overlap_ci
year_pair,,
2008-2009,"(0.5889, 0.8114)","(0.0, 0.2775)"
2009-2010,"(0.4529, 0.8397)","(0.0, 0.4)"
2010-2011,"(0.5949, 0.9221)","(0.0, 0.3)"


In [ ]:
# Cell 23: VALIDATION - Step 8: Summarize MLM Stability Evidence
# (Originally Cell 30)

import numpy as np

def summarize_stability_evidence(adjacent_year_metrics, mlm_confidence_intervals):
    """
    Summarizes stability based on adjacent year metrics and CIs.
    """
    stability_summary = []
    logging.info("Summarizing MLM stability evidence...")

    if not adjacent_year_metrics:
        logging.warning("  No adjacent year metrics for stability summarization.")
        return []

    for metrics in adjacent_year_metrics:
        year_pair = metrics["year_pair"]
        jsd = metrics.get("jensen_shannon_divergence")
        overlap = metrics.get("rank_overlap")
        cis = mlm_confidence_intervals.get(year_pair, {})

        if jsd is None or overlap is None or np.isnan(jsd) or np.isnan(overlap):
             assessment = "Inconclusive (Metric Error)"
             details = {"jsd": jsd, "rank_overlap": overlap, "jsd_ci": None, "rank_overlap_ci": None}
             logging.warning(f"  {year_pair}: {assessment}")
        else:
            jsd = float(jsd); overlap = float(overlap)
            jsd_ci = cis.get("jensen_shannon_divergence_ci")
            overlap_ci = cis.get("rank_overlap_ci")
            details = {"jsd": jsd, "rank_overlap": overlap, "jsd_ci": jsd_ci, "rank_overlap_ci": overlap_ci}

            is_unstable = False; reasons = []
            jsd_threshold_high = 0.1
            if jsd > jsd_threshold_high:
                is_unstable = True
                reasons.append(f"High JSD ({jsd:.3f} > {jsd_threshold_high})")
                if jsd_ci and jsd_ci[0] > 0.01:
                     reasons.append(f"JSD CI Low > 0.01")

            overlap_threshold_low = 0.5
            if overlap < overlap_threshold_low:
                is_unstable = True
                reasons.append(f"Low Overlap ({overlap:.3f} < {overlap_threshold_low})")
                if overlap_ci and overlap_ci[1] < 0.9:
                     reasons.append(f"Overlap CI High < 0.9")

            if is_unstable: assessment = f"Potential Shift ({'; '.join(reasons)})"
            else: assessment = "Stable"
            logging.info(f"  {year_pair}: JSD={jsd:.3f}, Overlap={overlap:.3f} -> {assessment}")

        stability_summary.append({
            "year_pair": year_pair,
            "stability_assessment": assessment,
            "details": details
        })

    return stability_summary

stability_assessments = summarize_stability_evidence(adjacent_year_metrics, mlm_confidence_intervals)

print("\n--- Semantic Stability Summary (MLM) ---")
if stability_assessments:
     try:
        import pandas as pd
        display(pd.DataFrame(stability_assessments).set_index('year_pair'))
     except ImportError:
        for assessment in stability_assessments: print(assessment)
else:
    print("No stability assessments generated.")


--- Semantic Stability Summary (MLM) ---


,stability_assessment,details
year_pair,,
2008-2009,Potential Shift (High JSD (0.672 > 0.1); JSD C...,"{'jsd': 0.6723, 'rank_overlap': 0.0, 'jsd_ci':..."
2009-2010,Potential Shift (High JSD (0.641 > 0.1); JSD C...,"{'jsd': 0.6414, 'rank_overlap': 0.1, 'jsd_ci':..."
2010-2011,Potential Shift (High JSD (0.730 > 0.1); JSD C...,"{'jsd': 0.7295, 'rank_overlap': 0.2, 'jsd_ci':..."


In [ ]:
# Cell 24: VALIDATION - Step 9: Run BERTopic
# (Originally Cell 21 - with corrected save function)

# Filter sentences and extract docs (use preprocessed_sentences_topic from Step 2)
pre_period_sentences_topic = [s for s in preprocessed_sentences_topic if s["year"] < midpoint_year]
post_period_sentences_topic = [s for s in preprocessed_sentences_topic if s["year"] >= midpoint_year]

pre_docs_topic = [s["processed_sentence"] for s in pre_period_sentences_topic]
post_docs_topic = [s["processed_sentence"] for s in post_period_sentences_topic]
pre_metadata_topic = pre_period_sentences_topic
post_metadata_topic = post_period_sentences_topic

logging.info(f"Using {len(pre_docs_topic)} docs for pre-period BERTopic.")
logging.info(f"Using {len(post_docs_topic)} docs for post-period BERTopic.")

# Install if needed
try:
  import bertopic
except ImportError:
  !pip install -q bertopic
from bertopic import BERTopic
import pandas as pd
import json
import os
from pathlib import Path
import logging
import numpy as np

# --- CORRECTED save_bertopic_results function ---
def save_bertopic_results(model, docs, doc_metadata, assigned_topics, output_prefix, out_dir):
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    logging.info(f"Saving BERTopic results for prefix '{output_prefix}' to {out_dir}")
    try:
        topic_info = model.get_topic_info()
    except Exception as e:
        logging.error(f"Could not get topic info for {output_prefix}: {e}")
        return
    if topic_info.empty or len(topic_info) <= 1:
        logging.warning(f"No significant topics found for {output_prefix}. Skipping detailed save.")
        results_json = {'topic_summary': [], 'yearly_distribution': {}}
        json_path = out_path / f"{output_prefix}.json"
        try:
           save_json_with_numpy(results_json, str(json_path))
           logging.info(f"Saved minimal JSON to: {json_path}")
        except Exception as save_err:
           logging.error(f"Failed to save minimal JSON: {save_err}")
        return

    try:
        rep_docs_dict = model.get_representative_docs()
        topic_info['Representative_Docs'] = topic_info['Topic'].apply(lambda x: list(rep_docs_dict.get(x, []))) # Ensure list
    except Exception as e:
        logging.warning(f"Could not get representative docs: {e}")
        topic_info['Representative_Docs'] = [[] for _ in range(len(topic_info))]

    yearly_distribution = pd.DataFrame()
    if not isinstance(doc_metadata, list) or not doc_metadata or not all('year' in d for d in doc_metadata):
         logging.error(f"doc_metadata invalid. Cannot calc yearly distribution.")
    else:
        docs_df = pd.DataFrame(doc_metadata)
        if len(docs_df) != len(assigned_topics):
             logging.warning(f"Length mismatch metadata ({len(docs_df)}) vs topics ({len(assigned_topics)}). Aligning.")
             min_len = min(len(docs_df), len(assigned_topics))
             docs_df = docs_df.iloc[:min_len].copy()
             current_assigned_topics = assigned_topics[:min_len]
             if len(docs_df) > 0: docs_df['Topic'] = current_assigned_topics
             else: docs_df = pd.DataFrame()
        elif len(docs_df) > 0 : docs_df['Topic'] = assigned_topics
        else: docs_df = pd.DataFrame()

        if not docs_df.empty and 'year' in docs_df.columns and 'Topic' in docs_df.columns:
            try:
                docs_df['year'] = docs_df['year'].astype(int)
                docs_df['Topic'] = docs_df['Topic'].astype(int)
                yearly_distribution = docs_df.groupby(['year', 'Topic']).size().unstack(fill_value=0)
            except Exception as e:
                logging.error(f"Error during groupby: {e}")
                yearly_distribution = pd.DataFrame()

    try:
        topic_summary_dict = topic_info.astype(str).to_dict('records')
        yearly_dist_dict = yearly_distribution.astype(int).to_dict('index') if not yearly_distribution.empty else {}
        results_json = {'topic_summary': topic_summary_dict, 'yearly_distribution': yearly_dist_dict}
    except Exception as e:
         logging.error(f"Error converting results to dict: {e}")
         results_json = {'topic_summary': [], 'yearly_distribution': {}}

    json_path = out_path / f"{output_prefix}.json"
    save_json_with_numpy(results_json, str(json_path)) # USE NUMPY SAFE SAVE

    csv_path = out_path / f"{output_prefix}_summary.csv"
    try:
        csv_topic_info = topic_info[['Topic', 'Count', 'Name', 'Representation']].copy()
        csv_topic_info['Representation'] = csv_topic_info['Representation'].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))
        csv_topic_info.to_csv(csv_path, index=False)
        logging.info(f"Saved topic summary CSV: {csv_path}")
    except Exception as e: logging.error(f"Failed to save CSV {csv_path}: {e}")

    if not yearly_distribution.empty:
        csv_dist_path = out_path / f"{output_prefix}_yearly_dist.csv"
        try:
            yearly_distribution.to_csv(csv_dist_path)
            logging.info(f"Saved yearly dist CSV: {csv_dist_path}")
        except Exception as e: logging.error(f"Failed to save CSV {csv_dist_path}: {e}")

# --- Train BERTopic Models ---
pre_model, post_model = None, None
pre_topics, post_topics = [], []

if pre_docs_topic:
    logging.info("Training BERTopic pre-period...")
    try:
        pre_model = BERTopic(verbose=False, min_topic_size=BERTOPIC_MIN_SIZE)
        pre_topics, _ = pre_model.fit_transform(pre_docs_topic)
        logging.info(f"Pre-period BERTopic complete. Found {len(set(pre_topics) - {-1})} topics.")
    except Exception as e:
        logging.error(f"BERTopic pre-period failed: {e}", exc_info=True)
        pre_model = None
else: logging.warning("No pre-period docs for BERTopic.")

if post_docs_topic:
    logging.info("Training BERTopic post-period...")
    try:
        post_min_size = max(2, min(BERTOPIC_MIN_SIZE, len(post_docs_topic) // 5)) if len(post_docs_topic) >= 10 else 2
        post_model = BERTopic(verbose=False, min_topic_size=post_min_size)
        post_topics, _ = post_model.fit_transform(post_docs_topic)
        logging.info(f"Post-period BERTopic complete (min_size={post_min_size}). Found {len(set(post_topics) - {-1})} topics.")
    except Exception as e:
        logging.error(f"BERTopic post-period failed: {e}", exc_info=True)
        post_model = None
else: logging.warning("No post-period docs for BERTopic.")

# --- Save Results ---
ts = datetime.now().strftime("%Y%m%d-%H%M%S")
if pre_model and pre_docs_topic:
    save_bertopic_results(pre_model, pre_docs_topic, pre_metadata_topic, pre_topics, f"topics_pre_{analysis_span[0]}-{analysis_span[1]}_{ts}", TOPIC_MODEL_DIR)
if post_model and post_docs_topic:
    save_bertopic_results(post_model, post_docs_topic, post_metadata_topic, post_topics, f"topics_post_{analysis_span[0]}-{analysis_span[1]}_{ts}", TOPIC_MODEL_DIR)

# --- Display Results ---
print("\n--- Pre-period Topic Info ---")
if pre_model:
    try: display(pre_model.get_topic_info())
    except Exception as e: print(f"Display error: {e}")
else: print("Pre-period model failed/unavailable.")

print("\n--- Post-period Topic Info ---")
if post_model:
    try: display(post_model.get_topic_info())
    except Exception as e: print(f"Display error: {e}")
else: print("Post-period model failed/unavailable.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 7.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.



--- Pre-period Topic Info ---


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1,-1_collisions_feshbach_resonances_ultracold,"[collisions, feshbach, resonances, ultracold, ...",[feshbach resonances mixtures ultracold gases ...
1,0,309,0_cloud_stars_star_formation,"[cloud, stars, star, formation, molecular, clu...",[multiwavelength study galactic hii region pre...
2,1,12,1_atoms_atomic_trap_cold,"[atoms, atomic, trap, cold, ultracold, optical...",[accumulation chromium metastable atoms optica...
3,2,8,2_polarized_bec_vortex_fermi,"[polarized, bec, vortex, fermi, lattice, conde...",[tkachenko modes superfluid fermi gas unitarit...



--- Post-period Topic Info ---


,Topic,Count,Name,Representation,Representative_Docs
0,-1,8,-1_dust_cloud_condensations_velocity,"[dust, cloud, condensations, velocity, particl...",[large aperture submillimeter telescope blast ...
1,0,4,0_star_formation_stellar_sources,"[star, formation, stellar, sources, young, seq...",[aste submillimeter observations young stellar...
2,1,3,1_sample_spectral_cloud_clouds,"[sample, spectral, cloud, clouds, dwarfs, data...",[eclipsing binary stars large small magellanic...


In [ ]:
# Cell 25: VALIDATION - Step 10: Match Topic Clusters
# (Originally Cell 22)

import numpy as np

def match_clusters(pre_model, post_model, similarity_threshold=0.1, min_overlap_keywords=2):
    """Matches clusters based on keyword overlap."""
    if pre_model is None or post_model is None:
        logging.error("One or both BERTopic models NA for matching.")
        return {"average_similarity": 0.0, "emergent_topics": [], "fading_topics": []}
    try:
        pre_topics_ids = list(pre_model.get_topics().keys())
        post_topics_ids = list(post_model.get_topics().keys())
    except Exception as e:
         logging.error(f"Error getting topic IDs: {e}")
         return {"average_similarity": 0.0, "emergent_topics": [], "fading_topics": []}

    pre_topics_keywords = {}
    for topic_id in pre_topics_ids:
        if topic_id != -1:
            try: pre_topics_keywords[topic_id] = [word for word, _ in pre_model.get_topic(topic_id)]
            except Exception as e: logging.warning(f"Could not get keywords for pre-topic {topic_id}: {e}")

    post_topics_keywords = {}
    for topic_id in post_topics_ids:
         if topic_id != -1:
            try: post_topics_keywords[topic_id] = [word for word, _ in post_model.get_topic(topic_id)]
            except Exception as e: logging.warning(f"Could not get keywords for post-topic {topic_id}: {e}")

    if not pre_topics_keywords: logging.warning("No non-outlier pre-topics found.")
    if not post_topics_keywords: logging.warning("No non-outlier post-topics found.")

    matched_similarities = []
    post_matched_status = {topic_id: False for topic_id in post_topics_keywords.keys()}
    fading_topics = list(pre_topics_keywords.keys())

    for post_topic_id, post_keywords in post_topics_keywords.items():
        if not post_keywords: continue
        best_match_similarity = -1
        matched_pre_topic = None
        for pre_topic_id, pre_keywords in pre_topics_keywords.items():
            if not pre_keywords: continue
            intersection = set(post_keywords) & set(pre_keywords)
            union = set(post_keywords) | set(pre_keywords)
            jaccard_similarity = len(intersection) / len(union) if union else 0
            if len(intersection) >= min_overlap_keywords and jaccard_similarity >= similarity_threshold:
                 if jaccard_similarity > best_match_similarity:
                      best_match_similarity = jaccard_similarity
                      matched_pre_topic = pre_topic_id
        if matched_pre_topic is not None:
             matched_similarities.append(best_match_similarity)
             post_matched_status[post_topic_id] = True
             if matched_pre_topic in fading_topics:
                 fading_topics.remove(matched_pre_topic)

    emergent_topics_details = [
        {"topic_id": post_topic_id, "keywords": post_topics_keywords[post_topic_id]}
        for post_topic_id, is_matched in post_matched_status.items() if not is_matched
    ]
    fading_topics_details = [
        {"topic_id": topic_id, "keywords": pre_topics_keywords[topic_id]}
        for topic_id in fading_topics if topic_id in pre_topics_keywords
    ]
    average_similarity = np.mean(matched_similarities) if matched_similarities else 0.0

    return {
        "average_similarity": float(average_similarity),
        "emergent_topics": emergent_topics_details,
        "fading_topics": fading_topics_details
    }

cluster_matching_results = match_clusters(pre_model, post_model)

print("\n--- Cluster Matching Results ---")
print(f"Average Similarity: {cluster_matching_results['average_similarity']:.3f}")
print("\nEmergent Topics:")
if cluster_matching_results['emergent_topics']:
    for topic in cluster_matching_results['emergent_topics']: print(f"  Post-Topic {topic['topic_id']}: {', '.join(topic['keywords'][:10])}")
else: print("  None found.")
print("\nFading Topics:")
if cluster_matching_results['fading_topics']:
    for topic in cluster_matching_results['fading_topics']: print(f"  Pre-Topic {topic['topic_id']}: {', '.join(topic['keywords'][:10])}")
else: print("  None found.")


--- Cluster Matching Results ---
Average Similarity: 0.144

Emergent Topics:
  None found.

Fading Topics:
  Pre-Topic 1: atoms, atomic, trap, cold, ultracold, optical, transport, superradiant, wave, coherence
  Pre-Topic 2: polarized, bec, vortex, fermi, lattice, condensates, condensate, dipolar, superfluid, phase


In [ ]:
# Cell 26: VALIDATION - Step 11: Combine Evidence
# (Originally Cell 29)

if 'cluster_matching_results' not in globals(): cluster_matching_results = {}
if 'adjacent_year_metrics' not in globals(): adjacent_year_metrics = []
if 'mlm_confidence_intervals' not in globals(): mlm_confidence_intervals = {}
if 'stability_assessments' not in globals(): stability_assessments = []

integrated_evidence = {
    "topic_reorganization": cluster_matching_results,
    "mlm_stability": {
        "adjacent_year_metrics": adjacent_year_metrics,
        "confidence_intervals": mlm_confidence_intervals,
        "stability_assessments": stability_assessments
    },
    "analysis_parameters": {
        "seed_word": SEED_WORD, "analysis_span": analysis_span,
        "validation_window": (window_start_year, window_end_year),
        "reference_year_drift": REFERENCE_YEAR, "genres_used": GENRES_TO_USE,
        "mlm_model": MLM_MODEL_NAME, "mlm_sentences_per_year": MLM_SENTENCES_PER_YEAR,
        "mlm_rank_overlap_k": MLM_RANK_OVERLAP_K, "mlm_bootstrap_samples": MLM_BOOTSTRAP_SAMPLES,
        "bertopic_min_size": BERTOPIC_MIN_SIZE,
        "topic_match_threshold": 0.1, "topic_match_min_keywords": 2,
        "w2v_vector_size": W2V_VECTOR_SIZE, "w2v_min_count": W2V_MIN_COUNT,
        "w2v_window": W2V_WINDOW, "bert_drift_model": BERT_MODEL_NAME,
        "topk_neighbors_drift": TOPK_NEIGHBORS
    }
}
print("Combined evidence structure 'integrated_evidence' created.")

Combined evidence structure 'integrated_evidence' created.


In [ ]:
# Cell 27: VALIDATION - Step 12: Summarize Overall Change Strength


import numpy as np

def summarize_overall_change_strength(integrated_evidence):
    topic_results = integrated_evidence.get("topic_reorganization", {})
    mlm_results = integrated_evidence.get("mlm_stability", {})

    overall_assessment = {
        "assessment": "Inconclusive",
        "explanation": "Evidence from methods needs comparison.",
        "details": {}
    }

    # Assess Topic Modeling
    avg_similarity = topic_results.get("average_similarity", None)
    if isinstance(avg_similarity, np.generic): avg_similarity = float(avg_similarity)
    emergent_topics = topic_results.get("emergent_topics", [])
    fading_topics = topic_results.get("fading_topics", [])

    topic_signal = "neutral"; topic_reasons = []
    if emergent_topics or fading_topics:
        topic_signal = "shift"
        topic_reasons.append("Topic reorganization detected.")
    if avg_similarity is not None:
        if avg_similarity < 0.5:
            if topic_signal != "shift": topic_signal = "shift"
            topic_reasons.append(f"Low avg topic similarity ({avg_similarity:.3f}).")
        elif avg_similarity >= 0.7 and not (emergent_topics or fading_topics):
             topic_signal = "stable"
             topic_reasons.append(f"High avg topic similarity ({avg_similarity:.3f}).")
        elif topic_signal != "shift":
             topic_reasons.append(f"Moderate avg topic similarity ({avg_similarity:.3f}).")
    if not topic_reasons and topic_signal == "neutral":
        topic_reasons.append("No strong topic signal detected.")

    # Assess MLM
    stability_assessments = mlm_results.get("stability_assessments", [])
    mlm_signal = "neutral"; mlm_reasons = []
    potential_shifts_mlm = [a for a in stability_assessments if "Potential Shift" in a.get("stability_assessment", "")]

    if potential_shifts_mlm:
        mlm_signal = "shift"
        mlm_reasons.append(f"MLM indicates potential shifts in: {[s['year_pair'] for s in potential_shifts_mlm]}.")
    # --- FIX: Corrected typo mllm -> mlm ---
    elif stability_assessments and not potential_shifts_mlm:
         mlm_signal = "stable"
         mlm_reasons.append("MLM indicates stable distributions.")
    elif not stability_assessments:
         mlm_reasons.append("No MLM stability assessments generated.")

    # Compare signals
    if topic_signal == "shift" and mlm_signal == "shift":
        overall_assessment["assessment"] = "Strong evidence for semantic shift"
        overall_assessment["explanation"] = "Both topic modeling and MLM indicate a shift."
    elif topic_signal == "stable" and mlm_signal == "stable":
        overall_assessment["assessment"] = "Strong evidence for semantic stability"
        overall_assessment["explanation"] = "Both topic modeling and MLM indicate stability."
    elif topic_signal == "shift" and mlm_signal != "shift":
        overall_assessment["assessment"] = "Partial Evidence for Shift (Topic Modeling)"
        overall_assessment["explanation"] = "Topic modeling indicates reorganization; MLM does not."
    elif topic_signal != "shift" and mlm_signal == "shift":
        overall_assessment["assessment"] = "Partial Evidence for Shift (MLM)"
        overall_assessment["explanation"] = "MLM indicates potential shifts; topic modeling does not."
    elif topic_signal == "neutral" and mlm_signal == "neutral":
         overall_assessment["assessment"] = "Inconclusive / Neutral Evidence"
         overall_assessment["explanation"] = "Neither method provided strong evidence."
    else: # Fallback
         overall_assessment["assessment"] = "Insufficient Data"
         overall_assessment["explanation"] = "Insufficient data/results for assessment."

    overall_assessment["details"] = {
        "topic_modeling_signal": topic_signal, "topic_modeling_reasons": topic_reasons,
        "mlm_signal": mlm_signal, "mlm_reasons": mlm_reasons
    }

    print("\n--- Overall Semantic Change Assessment ---")
    print(f"Assessment: {overall_assessment['assessment']}")
    print(f"Explanation: {overall_assessment['explanation']}")
    return overall_assessment

overall_change_assessment = summarize_overall_change_strength(integrated_evidence)


--- Overall Semantic Change Assessment ---
Assessment: Strong evidence for semantic shift
Explanation: Both topic modeling and MLM indicate a shift.


In [ ]:
# Cell 28: VALIDATION - Step 13: Generate Final Report Structure

import logging
from datetime import datetime

logging.info("Generating final report structure...")

if 'integrated_evidence' not in globals():
     logging.error("FATAL: 'integrated_evidence' not found!")
     integrated_evidence = {"analysis_parameters": {}, "topic_reorganization": {}, "mlm_stability": {}}
if 'overall_change_assessment' not in globals():
     logging.error("FATAL: 'overall_change_assessment' not found!")
     overall_change_assessment = {"assessment": "Error", "explanation": "Overall assessment failed."}

final_report = {
    "analysis_summary": {
        "seed_word": integrated_evidence.get("analysis_parameters", {}).get("seed_word", "N/A"),
        "analysis_span": integrated_evidence.get("analysis_parameters", {}).get("analysis_span", "N/A"),
        "overall_assessment": overall_change_assessment.get("assessment", "Error"),
        "assessment_explanation": overall_change_assessment.get("explanation", "N/A")
    },
    "topic_modeling_results": integrated_evidence.get("topic_reorganization", {}),
    "mlm_analysis_results": integrated_evidence.get("mlm_stability", {}),
    "analysis_parameters": integrated_evidence.get("analysis_parameters", {}),
    "provenance": {
        "report_generated_on": datetime.now().isoformat(),
        "note": "Integrated report combining Topic Modeling and MLM stability analysis results."
    }
}
logging.info("Final report structure created.")
print("\n--- Final Report Preview ---")
print("Analysis Summary:", final_report.get("analysis_summary", {}))
print("Topic Keys:", list(final_report.get("topic_modeling_results", {}).keys()))
print("MLM Keys:", list(final_report.get("mlm_analysis_results", {}).keys()))


--- Final Report Preview ---
Analysis Summary: {'seed_word': 'cloud', 'analysis_span': (2008, 2011), 'overall_assessment': 'Strong evidence for semantic shift', 'assessment_explanation': 'Both topic modeling and MLM indicate a shift.'}
Topic Keys: ['average_similarity', 'emergent_topics', 'fading_topics']
MLM Keys: ['adjacent_year_metrics', 'confidence_intervals', 'stability_assessments']


In [ ]:
# Cell 29: VALIDATION - Step 14: Save Final Report


import os
import json
from datetime import datetime
import numpy as np
import pandas as pd

# Ensure numpy-safe save function exists (defined in Cell 13)
if 'save_json_with_numpy' not in globals():
     logging.error("FATAL: save_json_with_numpy function not found. Report saving will fail.")
     # Define it here as a fallback
     def save_json_with_numpy(obj, path):
        def default_converter(o):
            if isinstance(o, (np.float32, np.float64)): return float(o)
            if isinstance(o, (np.int32, np.int64)): return int(o)
            if isinstance(o, np.ndarray): return o.tolist()
            if isinstance(o, (datetime, pd.Timestamp)): return o.isoformat()
            raise TypeError(f"Object of type {o.__class__.__name__} is not JSON serializable")
        try:
            with open(path, "w", encoding="utf-8") as f:
                json.dump(obj, f, ensure_ascii=False, indent=2, default=default_converter)
            logging.info(f"Saved (fallback function): {path}")
        except Exception as e:
            logging.error(f"Error saving {path} (fallback): {e}", exc_info=True)

# --- Use VALIDATION_REPORT_DIR from Cell 1 ---
OUT_DIR_REPORT = VALIDATION_REPORT_DIR
os.makedirs(OUT_DIR_REPORT, exist_ok=True)
logging.info(f"Saving final validation report to: {OUT_DIR_REPORT}")

seed = SEED_WORD if 'SEED_WORD' in globals() else "analysis"
ts_report = datetime.now().strftime("%Y%m%d-%H%M%S")
report_filename = f"semanticChangeReport_{seed}_{ts_report}.json"
report_path = os.path.join(OUT_DIR_REPORT, report_filename)

try:
    save_json_with_numpy(final_report, report_path)
    print(f"Final validation report saved successfully: {report_path}")
except NameError:
     print("Error: 'final_report' variable not defined. Ensure Step 13 ran successfully.")
except Exception as e:
    print(f"Error saving final validation report: {e}")

print(f"\n--- Contents of {OUT_DIR_REPORT} ---")
!ls -lht "{OUT_DIR_REPORT}" | head -n 10

Final validation report saved successfully: /content/drive/MyDrive/Projects/Upwork/outputs/validation_reports/semanticChangeReport_cloud_20251101-080809.json

--- Contents of /content/drive/MyDrive/Projects/Upwork/outputs/validation_reports ---
total 5.5K
-rw------- 1 root root 4.1K Nov  1 08:08 semanticChangeReport_cloud_20251101-080809.json
-rw------- 1 root root  541 Oct 25 16:24 semanticChangeReport_cloud_20251025-162407.json
